# Experiments

This file contains multiple experiments that were done in order to find the solution to the [task](../task.md). Most of the necesary code is located in [solution.py](solution.py).

## Loading the training data

In [ ]:
# %pip uninstall torch torchvision torchaudio -y
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# %pip install scikit-learn numpy

In [1]:
from solution import read_trainset

classMap, train_data = read_trainset()
train_data

Dataset ImageFolder
    Number of datapoints: 74490
    Root location: c:\Users\hanna\OneDrive\PROGRAMOWANIE\Python\SSNE\mini_project_3\train
    StandardTransform
Transform: Compose(
               Resize(size=(48, 48), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

## Method of evaluation
In order to make sure that the following experiments are reliable we used cross-validation for measuring prediction quality. The accuracy itself is measured just like in the [helpers.py](../helpers.py) file provided.

In [2]:
from torch.utils.data import Subset, DataLoader
from sklearn.model_selection import KFold
from solution import NeuralNetwork
import numpy as np
import copy


def calc_accuracy(predictions: np.ndarray, targets: np.ndarray, n_classes=50):
    assert len(predictions) == len(targets)
    accuracies = []
    for i in range(n_classes):
        mask = targets == i
        n_samples = mask.sum()
        if n_samples > 0:
            accuracies.append((predictions[mask] == i).sum() / n_samples)
        else:
            accuracies.append(0.0)  # Klasa nieobecna → accuracy = 0 zamiast NaN
    return np.mean(accuracies)


def evaluate_cross_validation(experiment_name, model_instance, full_dataset, number_of_folds=3):
    kfold = KFold(n_splits=number_of_folds, shuffle=True, random_state=42)
    accuracy_scores = []
    dataset_indices = np.arange(len(full_dataset))
    initial_state = model_instance.model.state_dict()
    
    for fold_number, (train_indices, validation_indices) in enumerate(kfold.split(dataset_indices)):
        model_copy = copy.deepcopy(model_instance)  
        model_copy.model.load_state_dict(initial_state)
        
        train_dataloader = DataLoader(
            Subset(full_dataset, train_indices), 
            batch_size=model_copy.config.batch_size, 
            shuffle=True,
            num_workers=4,              
            pin_memory=torch.cuda.is_available(),          
            prefetch_factor=2,       
            persistent_workers=True     
        )
        validation_dataloader = DataLoader(
            Subset(full_dataset, validation_indices), 
            batch_size=model_copy.config.batch_size, 
            shuffle=False,
            num_workers=4,
            pin_memory=torch.cuda.is_available(),
            prefetch_factor=2,
            persistent_workers=True
        )
        
        model_copy.fit(train_dataloader)        
        predictions = np.array([result[1] for result in model_copy.predict(validation_dataloader)])
        true_labels = np.array([full_dataset.targets[i] for i in validation_indices])
        
        fold_accuracy = calc_accuracy(predictions, true_labels)
        accuracy_scores.append(fold_accuracy)

    mean_accuracy = np.mean(accuracy_scores)
    std_deviation = np.std(accuracy_scores)
    print(f"\n{experiment_name} Results: {mean_accuracy:.4f} ± {std_deviation:.4f}")

def compare_models(experiments, training_data=train_data):
    '''using a list of experiments and models runs CV on all'''

    for experiment_name, config in experiments:
        model = NeuralNetwork(config)
        evaluate_cross_validation(experiment_name, model, train_data)

## Experiments

In [3]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"Device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.5.1+cu121
CUDA available: True
Device: NVIDIA GeForce RTX 4060 Laptop GPU


### Batch Normalization & Batch sizes

In [ ]:
from solution import NetConfig

bn_batch_experiments = [
    ("No BN + Batch=128", NetConfig(use_batch_normalization=False, batch_size=128, epochs=7)),
    ("No BN + Batch=256", NetConfig(use_batch_normalization=False, batch_size=256, epochs=7)),
    ("BN + Batch=64",     NetConfig(use_batch_normalization=True,  batch_size=64, epochs=7)),
    ("BN + Batch=128",    NetConfig(use_batch_normalization=True,  batch_size=128, epochs=7)),
    ("BN + Batch=256",    NetConfig(use_batch_normalization=True,  batch_size=256, epochs=7))
]

compare_models(bn_batch_experiments)

As expected batch normalization significantly improves the results.

### Net layers and depth

In [ ]:
depth_experiments = [
    ("Shallow", NetConfig(convolutional_layers=[32,64], fully_connected_layers=[256], epochs=2, use_batch_normalization=True)),
    ("Medium",  NetConfig(convolutional_layers=[32,64,128], fully_connected_layers=[512,128], epochs=2, use_batch_normalization=True)),
    ("Deep",    NetConfig(convolutional_layers=[32,64,128,256], fully_connected_layers=[512,256,128], epochs=2, use_batch_normalization=True))
]

compare_models(depth_experiments)

------------------------------
\tEpoch 1/2 - Loss: 2.7507, Accuracy: 26.47%
\tEpoch 2/2 - Loss: 2.2455, Accuracy: 38.31%
\tEpoch 1/2 - Loss: 2.7194, Accuracy: 27.51%
\tEpoch 2/2 - Loss: 2.1999, Accuracy: 39.49%

Shallow Results: 0.3247 ± 0.0067
------------------------------
\tEpoch 1/2 - Loss: 2.9393, Accuracy: 22.63%
\tEpoch 2/2 - Loss: 2.3622, Accuracy: 34.93%
\tEpoch 1/2 - Loss: 2.9087, Accuracy: 23.63%
\tEpoch 2/2 - Loss: 2.3115, Accuracy: 36.41%

Medium Results: 0.3357 ± 0.0021
------------------------------
\tEpoch 1/2 - Loss: 3.0583, Accuracy: 18.56%
\tEpoch 2/2 - Loss: 2.5050, Accuracy: 30.40%
\tEpoch 1/2 - Loss: 3.0468, Accuracy: 18.75%
\tEpoch 2/2 - Loss: 2.4941, Accuracy: 30.62%

Deep Results: 0.2999 ± 0.0038


### Dropout

In [ ]:
dropout_experiments = [
    ("No Dropout",  NetConfig(dropout_rates=[0.0], epochs=2, use_batch_normalization=True)),
    ("Moderate",    NetConfig(dropout_rates=[0.2], epochs=2, use_batch_normalization=True)),
    ("Strong",      NetConfig(dropout_rates=[0.4], epochs=2, use_batch_normalization=True))
]
compare_models(dropout_experiments)

------------------------------
\tEpoch 1/2 - Loss: 2.7320, Accuracy: 27.61%
\tEpoch 2/2 - Loss: 2.0954, Accuracy: 41.88%
\tEpoch 1/2 - Loss: 2.7083, Accuracy: 28.04%
\tEpoch 2/2 - Loss: 2.0705, Accuracy: 42.29%

No Dropout Results: 0.3434 ± 0.0008
------------------------------
\tEpoch 1/2 - Loss: 2.8441, Accuracy: 23.92%
\tEpoch 2/2 - Loss: 2.2425, Accuracy: 37.99%
\tEpoch 1/2 - Loss: 2.8294, Accuracy: 24.88%
\tEpoch 2/2 - Loss: 2.2222, Accuracy: 38.71%

Moderate Results: 0.3291 ± 0.0013
------------------------------
\tEpoch 1/2 - Loss: 3.0247, Accuracy: 19.01%
\tEpoch 2/2 - Loss: 2.4771, Accuracy: 31.27%
\tEpoch 1/2 - Loss: 2.9953, Accuracy: 20.45%
\tEpoch 2/2 - Loss: 2.4343, Accuracy: 32.81%

Strong Results: 0.3058 ± 0.0004


### Optimizer

In [1]:
# ADAM, SGD (JAKIE MOMENTUM?), ADAMW

### Activation Functions

In [9]:
# Relu, linear, leaky Relu

### Augmentation


In [ ]:
# none, strong, weak
# wymaga wykorzystania na nowo read_trainset i tam wybrac none/weak/strong augmentation i ewentualnie ulepszyc te konfiguracje augmentacji

### Final Grid search

In [ ]:
# find best combo knowing the results of prev experiments

## Conclusions
During these experiments we found that the best model configuration was:
> ...

And that is the configuration we used to create our final predictions.
